In [1]:
from phase_II.fast_wigner_function.utils import *
import nifty.nifty.cl as ift
%matplotlib tk

In [26]:
from scipy.interpolate import interp1d


def interpolator(x_array, y_array):
    # if outside of bound, will be filled with 0's
    return interp1d(x_array, y_array, kind='cubic', bounds_error=False, fill_value=0)

def Stress(xi, frequencies, supress_print=False):
    """
    :param xi:                  The xi array in Fourier space, in symmetric ordering.
    :param frequencies:         The frequency values at which xi is reported, in symmetric ordering.
    :param supress_print:       If True, supress printing of the interpolated values.
    :return:
    """
    N = len(frequencies)
    f = frequencies
    # k = 2*f.copy()  # double to get exactly matching indices but effectively halve t_dual.
    k = f.copy()  # let the interpolator do the job
    dk = k[1]-k[0]
    dt = 1.0 / (N * dk)
    t_dual = np.arange(N) * dt  # k-dual time will have different spacing.


    # xi = CallableArray(x=f, y=xi, bounds_error=False, bound_min=np.min(f), bound_max=np.max(f))
    xi = interpolator(x_array=f, y_array=xi)
    f_cast, k_cast = (f[:, None], k[None, :])  # f are rows, k are columns
    Phi = xi(f_cast+1/2*k_cast).conj() * xi(f_cast-1/2*k_cast)

    Phi = np.fft.ifftshift(Phi, axes=1)  # apply DFT order to columns

    S_mat = np.fft.ifft(Phi, axis=1, norm='forward', n=N)
    S_mat = np.fft.fftshift(S_mat, axes=0)  # Reorder the rows (freqs) in the zero-centered convention for plotting.
    # Columns (times from iFFT) are already monotonically increasing in time since that's the standard order in real space.
    S_mat = np.fft.fftshift(S_mat, axes=1)

    if not supress_print:
        check_if_mat_imag(S_mat)

    f_zero_centered = np.fft.fftshift(f) # zero-centered freqs for plotting
    return S_mat, t_dual, f_zero_centered


def Stress_gpt(xi, frequencies, interpolator, supress_print=False):
    """
    Robust Stress computation returning (S_mat, t_dual, f_zero_centered).

    - Accepts xi and frequencies either in DFT standard order (0,+,..,-..) OR
      in zero-centered symmetric order (-..,0,..+). The function auto-normalizes
      them to symmetric order internally.
    - Uses `interpolator(x_array=f, y_array=xi)` if provided, else expects
      `interpolator` to be callable already; otherwise it will fallback to
      numpy.interp for reals (but you passed your own).
    """
    # --- basic checks & normalize arrays ---
    xi = np.asarray(xi)
    frequencies = np.asarray(frequencies)
    N = len(frequencies)
    if len(xi) != N:
        raise ValueError("xi and frequencies must have same length N")

    # detect order: if first element < 0 assume symmetric (fftshifted),
    # otherwise assume DFT standard order (0, +..., -...).
    is_symmetric = (frequencies[0] < 0)
    if is_symmetric:
        f_sym = frequencies.copy()
        xi_sym = xi.copy()
    else:
        # frequencies in DFT order -> convert to symmetric physical order
        f_sym = np.fft.fftshift(frequencies)
        xi_sym = np.fft.fftshift(xi)

    # check uniform spacing
    df = np.median(np.diff(f_sym))
    if not np.allclose(np.diff(f_sym), df, rtol=1e-6, atol=1e-12):
        raise ValueError("frequencies must be uniformly spaced")

    # --- build k grid (use same spacing as f) ---
    # use k = f (let interpolator handle off-grid evaluation) to avoid forced halving
    f = f_sym                       # rows (symmetric)
    k = f.copy()                    # columns (symmetric)
    dk = k[1] - k[0]
    if not np.isfinite(dk) or dk == 0:
        raise ValueError("invalid dk")
    # time dual spacing for kernel exp(2π i k t)
    dt = 1.0 / (N * dk)
    t_dual = np.arange(N) * dt      # standard order: 0... (monotonic)

    try:
        xi_call = interpolator(x_array=f, y_array=xi_sym)
    except TypeError:
        # if user passed a callable already
        xi_call = interpolator

    # --- build Phi vectorized (f as rows, k as columns) ---
    f_cast = f[:, None]   # (N,1)
    k_cast = k[None, :]   # (1,N)
    arg_plus  = f_cast + 0.5 * k_cast
    arg_minus = f_cast - 0.5 * k_cast

    # evaluate interpolator on whole grid
    Xi_plus  = xi_call(arg_plus)   # shape (N,N)
    Xi_minus = xi_call(arg_minus)

    Phi = Xi_plus.conj() * Xi_minus

    # --- k -> t transform (careful with order) ---
    # Phi currently has k in symmetric physical order (matching f).
    # np.fft.ifft expects DFT order, so swap to DFT order along axis=1,
    # ifft, then invert the swap to get time in standard order.
    Phi_dft = np.fft.fftshift(Phi, axes=1)         # symmetric -> DFT order
    S_ifft = np.fft.ifft(Phi_dft, axis=1, norm='forward', n=N)
    # after ifft, columns are in standard t order but still in DFT index order;
    # converting back:
    # S_mat = np.fft.ifftshift(S_ifft, axes=1)       # columns -> standard t order (monotonic)

    # rows are currently in symmetric f order (f), ready for plotting. If you prefer DFT
    # order return np.fft.ifftshift(f) and np.fft.fftshift(S_mat, axes=0).
    # We will return f_zero_centered = f (symmetric).
    f_zero_centered = f.copy()

    # optional realness check
    if not supress_print:
        diagnostic = np.abs(np.mean(S_mat.imag))
        if diagnostic < 1e-10:
            print(f"\u2714 Mean imaginary part of stress < 1e-10 ({diagnostic})")
        else:
            warnings.warn(f"Mean imaginary part of stress > 1e-10 ({diagnostic})")

    return S_mat, t_dual, f_zero_centered


In [27]:
L = 2
N = 2000
dx = L/N

x = ift.RGSpace(shape=(N,), distances=dx)
y = x.get_default_codomain()
t = np.linspace(0, L, N, endpoint=False)

f_nyquist = 1/(2*dx)
freqs_zero_centered = mirror_negative_frequencies(y.get_unique_k_lengths(), unique_k_lengths=True, standard_order=False)

In [28]:
star_idx = 150
xi_tilde = xi_field(case=1, N=N, omegas=freqs_zero_centered, peak_amplitude=1e6, peak_frequency=freqs_zero_centered[star_idx])


Constructing xi field for case 1: Spike

	f star:  -425.0  at index  150


In [29]:
freqs_reordered = np.fft.ifftshift(freqs_zero_centered)

spike_in_harmonic_space = False
if spike_in_harmonic_space:
    xi_tilde_reordered = np.fft.ifftshift(xi_tilde)
else:
    # extra FFT to simulate a spike in time instead
    xi_tilde = np.fft.fft(xi_tilde)  # is already ordered in standard DFT...
    xi_tilde_reordered = np.fft.ifftshift(xi_tilde)  # so doing it again doesn't hurt

    # plt.plot(freqs_zero_centered, np.fft.fftshift(xi_tilde_reordered).real)
    # plt.show()
    print(f"Time spike at idx {star_idx} corresponds to t = ", t[star_idx])

Time spike at idx 150 corresponds to t =  0.15


In [30]:
# S_mat, t_dual, f_zero_centered = Stress_gpt(xi=xi_tilde_reordered, frequencies=freqs_reordered, interpolator=interpolator)
S_mat, t_dual, f_zero_centered = Stress(xi=xi_tilde, frequencies=freqs_zero_centered)

In [31]:
visualize_stress(S_mat.real, rows=f_zero_centered, cols=t_dual)

In [32]:
standard_gaussian = xi_field(case=4, N=N, length=np.sqrt(L), use_complex=True)


Constructing xi field for case 4: Normal standard variable



In [33]:
stress_sn_normal, t, f = Stress(xi=standard_gaussian, frequencies=freqs_zero_centered)

✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (2.680337158254188e-19) 


In [34]:
visualize_stress(stress_sn_normal.real, rows=f, cols=t)

In [35]:
print("np mean ", np.mean(stress_sn_normal.real), np.std(stress_sn_normal.real))

np mean  0.9540540664695274 32.49840875147096


In [37]:
S_mat_collection = []
for _ in range(100):
    standard_gaussian = xi_field(case=4, N=N, length=np.sqrt(L), use_complex=True, supress_print=True)
    stress_sn_normal, _, _ = Stress(xi=standard_gaussian, frequencies=freqs_zero_centered, supress_print=True)
    S_mat_collection.append(stress_sn_normal)

S_mat_collection = np.array(S_mat_collection)

In [38]:
S_mat_average = np.mean(S_mat_collection.real, axis=0)
S_mat_std = np.std(S_mat_collection.real, axis=0)

visualize_stress(S_mat_average.real, rows=f_zero_centered, cols=t)

print("Mean and mean std: ", np.mean(S_mat_average.real), np.mean(S_mat_std.real))


Mean and mean std:  1.0008825307737172 30.601738761556796
